In [2]:
from helper import *
import json
from collections import defaultdict

def get_sailor_first_year(G, sailor_node):
    """Return the earliest release year among Sailor's works."""
    sailor_works, _ = find_all_artist_works(G, sailor_node)
    years = []
    for w in sailor_works:
        date = G.nodes[w].get('release_date')
        if date:
            years.append(int(date))
    return min(years) if years else None

def generate_sankey_influence_to_oceanus(G, sailor_start_year, output_path):
    """
    Build a Sankey diagram JSON showing flows from source genres to Oceanus Folk,
    split into two periods: before and after Sailor's career start.
    """
    influence_work_types = ['InStyleOf', 'CoverOf', 'InterpolatesFrom', 'DirectlySamples', 'LyricalReferenceTo']
    person_work_types = ['PerformerOf', 'ComposerOf', 'ProducerOf', 'LyricistOf']
    member_type = 'MemberOf'

    genre_before = defaultdict(int)
    genre_after = defaultdict(int)
    artist_before = defaultdict(int)
    artist_after = defaultdict(int)

    for node, attrs in G.nodes(data=True):
        node_type = attrs.get('Node Type')
        if node_type not in ('Song', 'Album'):
            continue
        if attrs.get('genre') != 'Oceanus Folk':
            continue

        year_str = attrs.get('release_date')
        if not year_str:
            continue
        target_year = int(year_str)

        period = 'before' if target_year < sailor_start_year else 'after'

        for etype in influence_work_types:
            for influencer_work in get_neighbors_by_edge_type(G, node, etype, 'in'):
                influencer_genre = G.nodes[influencer_work].get('genre')
                if not influencer_genre or influencer_genre == 'Oceanus Folk':
                    continue

                if period == 'before':
                    genre_before[influencer_genre] += 1
                else:
                    genre_after[influencer_genre] += 1

                # (Optional) count artists of the influencer work
                for ctype in person_work_types:
                    for artist in get_neighbors_by_edge_type(G, influencer_work, ctype, 'in'):
                        node_type_artist = G.nodes[artist].get('Node Type')
                        if node_type_artist == 'MusicalGroup':
                            for member in get_neighbors_by_edge_type(G, artist, member_type, 'in'):
                                member_name = G.nodes[member].get('name')
                                if period == 'before':
                                    artist_before[member_name] += 1
                                else:
                                    artist_after[member_name] += 1
                        else:
                            artist_name = G.nodes[artist].get('name')
                            if period == 'before':
                                artist_before[artist_name] += 1
                            else:
                                artist_after[artist_name] += 1

    # Build nodes for Sankey
    source_nodes = sorted(set(genre_before.keys()) | set(genre_after.keys()))
    nodes = [{"name": g} for g in source_nodes]
    nodes.append({"name": "Oceanus Folk (before)"})
    nodes.append({"name": "Oceanus Folk (after)"})

    node_index = {node["name"]: i for i, node in enumerate(nodes)}
    before_idx = node_index["Oceanus Folk (before)"]
    after_idx = node_index["Oceanus Folk (after)"]

    links = []
    for src_genre in source_nodes:
        before_cnt = genre_before.get(src_genre, 0)
        after_cnt = genre_after.get(src_genre, 0)
        if before_cnt > 0:
            links.append({"source": node_index[src_genre], "target": before_idx, "value": before_cnt})
        if after_cnt > 0:
            links.append({"source": node_index[src_genre], "target": after_idx, "value": after_cnt})

    output_data = {"nodes": nodes, "links": links}
    with open(output_path, "w") as f:
        json.dump(output_data, f, indent=2)

    print(f"Sankey JSON saved to {output_path}")
    print(f"Found {len(source_nodes)} source genres influencing Oceanus Folk.")
    print(f"Before Sailor: {sum(genre_before.values())} influence events")
    print(f"After Sailor: {sum(genre_after.values())} influence events")

def main():
    G = read_data_from_json_to_graph(data_file_path)
    sailor_node, _ = find_sailor_shift(G)
    if not sailor_node:
        print("Sailor Shift not found!")
        return
    sailor_start_year = get_sailor_first_year(G, sailor_node)
    print(f"Sailor Shift first appearance year: {sailor_start_year}")

    generate_sankey_influence_to_oceanus(G, sailor_start_year, "data/influence_to_oceanus_sankey.json")

if __name__ == "__main__":
    main()

Type of data: <class 'dict'>
Sailor Shift first appearance year: 2024
Sankey JSON saved to data/influence_to_oceanus_sankey.json
Found 16 source genres influencing Oceanus Folk.
Before Sailor: 192 influence events
After Sailor: 38 influence events
